#### **Step 1: Import Required Libraries**


In [15]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, abs, current_timestamp)
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 17, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [16]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/kafka_eventstream/raw_stream/betting_events"
df_raw_betting_events = spark.read.format("parquet").load(raw_transaction_path)
display(df_raw_betting_events.limit(10))

StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 41453679-971b-4d1b-b4cc-15e1730ed6f7)

### **Column-by-Column Transformation Plan**
user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

match_id
- Current: Clean and consistent (e.g., MATCH_773).
- Action: No transformation needed.
- Final: Keep as-is.

event_id
- Current: Clean and consistent (e.g., 68fae11e-c393-44ec-a35e-7fc6d9351629).
- Action: No transformation needed.
- Final: Keep as-is.

device_id
- Current: Clean and consistent (e.g., 68fae11e-c393-44ec-a35e-7fc6d9351629).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the bet_results dataframe for data quality checks**

In [17]:
# Select specific columns and count nulls
null_counts = df_raw_betting_events.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['event_id', 'user_id', 'match_id', 'bet_type', 'stake_amount', 'odds', 'bet_time', 'device_id', 'channel']
    ]
)

# Display results
null_counts.show()

StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 19, Finished, Available, Finished)

+--------+-------+--------+--------+------------+----+--------+---------+-------+
|event_id|user_id|match_id|bet_type|stake_amount|odds|bet_time|device_id|channel|
+--------+-------+--------+--------+------------+----+--------+---------+-------+
|       0|      0|       0|       0|        1490|1633|       0|        0|      0|
+--------+-------+--------+--------+------------+----+--------+---------+-------+



#### **Step 4: Data Transformation to channel column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels (Web, Mobile, POS).
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid channels (Web, Mobile, POS) using a canonical character signature map.
- Returned standardized labels: Web, Mobile, POS.
- Labeled unmatched or null values as "Unknown".

In [18]:
df_cleaned_betting_events = df_raw_betting_events
# df_cleaned_betting_events.select("channel").distinct().show(5)

# 1. Pre‑compute canonical signatures

def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a word."""
    return "".join(sorted(word.lower()))

# Define the valid channel values
canonical_values = ["Web", "Retail", "Mobile"]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)


# 2. UDF to normalize jumbled channels

@udf(StringType())
def normalize_channel(raw: str) -> str:
    if raw is None:
        return None  # Return None to allow dropping later
    signature = "".join(sorted(raw.lower().strip()))
    return sig_broadcast.value.get(signature)


# 3. Apply transformation

df_cleaned_betting_events = df_cleaned_betting_events.withColumn(
    "channel",
    normalize_channel(col("channel"))
)


# 4. Drop rows with unrecognized/null channel values

df_cleaned_betting_events = df_cleaned_betting_events.filter(col("channel").isNotNull())


# 5. Inspect cleaned channel values

df_cleaned_betting_events.select("channel").distinct().show(truncate=False)


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 20, Finished, Available, Finished)

+-------+
|channel|
+-------+
|Mobile |
|Retail |
|Web    |
+-------+



#### **Step :5 Data Transformation to bet_type column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels (Draw, HT/FT, Over/Under, Win Full-time).
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid channels (Draw, HT/FT, Over/Under, Win Full-time) using a canonical character signature map.
- Returned standardized labels: Draw, HT/FT, Over/Under, Win Full-time.
- Labeled unmatched or null values as "Unknown".

In [19]:
# df_cleaned_betting_events.select("bet_type").distinct().show()

# 1. Pre‑compute canonical signatures

def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a word."""
    return "".join(sorted(word.lower()))

# Define the valid channel values
canonical_values = ["Draw", "HT/FT", "Over/Under", "Win Full-time"]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)


# 2. UDF to normalize jumbled channels

@udf(StringType())
def normalize_channel(raw: str) -> str:
    if raw is None:
        return None  # Return None to allow dropping later
    signature = "".join(sorted(raw.lower().strip()))
    return sig_broadcast.value.get(signature)


# 3. Apply transformation

df_cleaned_betting_events = df_cleaned_betting_events.withColumn(
    "bet_type",
    normalize_channel(col("bet_type"))
)


# 4. Drop rows with unrecognized/null bet_type values

df_cleaned_betting_events = df_cleaned_betting_events.filter(col("bet_type").isNotNull())


# 5. Inspect cleaned bet_type values

df_cleaned_betting_events.select("bet_type").distinct().show(truncate=False)


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 21, Finished, Available, Finished)

+-------------+
|bet_type     |
+-------------+
|Draw         |
|Win Full-time|
|Over/Under   |
|HT/FT        |
+-------------+



#### **Step 6: Data Transformation to match_id column**
Issue:
- Some values had inconsistent casing like MATCH_867, match_432, or mixed-case variants.

Transformation:
- Converted all match_id values to lowercase.
- Trimmed any leading or trailing whitespace.

In [20]:
# Standardize `match_id` by converting to lowercase and trimming whitespace
df_cleaned_betting_events = df_cleaned_betting_events.withColumn(
    "match_id",
    lower(trim(col("match_id").cast("string")))
)

# Show sample records after transformation
df_cleaned_betting_events.select("match_id").show(5, truncate=False)


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 22, Finished, Available, Finished)

+--------+
|match_id|
+--------+
|match_92|
|match_8 |
|match_8 |
|match_8 |
|match_92|
+--------+
only showing top 5 rows



#### **Step 7: Data Transformation to bet_time column**
Issue:
- The column values were in ISO 8601 string format (yyyy-MM-ddTHH:mm:ssZ), not usable for date operations.

Transformation:
- Used to_timestamp() to convert string to proper timestamp format.
- Ensured time zone marker 'Z' was treated as a literal.

In [21]:
# Convert ISO 8601 format to timestamp
df_cleaned_betting_events = df_cleaned_betting_events.withColumn(
    "bet_time",
    to_timestamp(col("bet_time"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Show sample converted values
df_cleaned_betting_events.select("bet_time").show(5, truncate=False)


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 23, Finished, Available, Finished)

+-------------------+
|bet_time           |
+-------------------+
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
|2025-07-05 08:35:19|
+-------------------+
only showing top 5 rows



In [22]:
# Step 1: Remove rows where stake_amount is null
df_cleaned_betting_events = df_cleaned_betting_events.filter(col("stake_amount").isNotNull())

# Step 2: Convert stake_amount to positive and round to 0 decimal places
df_cleaned_betting_events = df_cleaned_betting_events.withColumn(
    "stake_amount",
    round(abs(col("stake_amount")), 0)
)

# Step 3: Keep only rows where stake_amount > 10
df_cleaned_betting_events = df_cleaned_betting_events.filter(col("stake_amount") > 10)

# Optional: Show the result
# df_cleaned_betting_events.select("stake_amount").show(5, truncate=False)


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 24, Finished, Available, Finished)

#### **Step 9: Data Transformation to odds column**
Issue:
- Some odds values were negative, which is inconsistent with expected business logic (odds should always be non-negative).
- Some values had decimal precision, whereas odds are typically represented as whole numbers.
- Some records contained null values, which are invalid for record fields like odds.

Transformation:
- Converted all odds values to positive using the abs() function.
- Rounded odds values to whole numbers using round(..., 0).
- Removed records where odds was null to ensure data integrity.

In [23]:
# Step 1: Remove rows where odds is null
df_cleaned_betting_events = df_cleaned_betting_events.filter(col("odds").isNotNull())

# Step 2: Convert odds to positive and round to 0 decimal places
df_cleaned_betting_events = df_cleaned_betting_events.withColumn(
    "odds",
    round(abs(col("odds")), 0)
)

# Step 3: Keep only rows where odds > 1
df_cleaned_betting_events = df_cleaned_betting_events.filter(col("odds") > 1)

# Optional: Show the result
# df_cleaned_betting_events.select("odds").show(10, truncate=False)
display(df_cleaned_betting_events.show(5))


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 25, Finished, Available, Finished)

+--------------------+----------+--------+-------------+------------+----+-------------------+--------------------+-------+----------+
|            event_id|   user_id|match_id|     bet_type|stake_amount|odds|           bet_time|           device_id|channel|event_date|
+--------------------+----------+--------+-------------+------------+----+-------------------+--------------------+-------+----------+
|d30ad7cb-634e-4cc...|BB10003525|match_92|        HT/FT|      4700.0| 4.0|2025-07-05 08:35:19|345104bb-41e5-4a4...| Mobile|2025-07-05|
|3588c5bb-bae9-4b2...|BB10002382| match_8|Win Full-time|      3885.0| 3.0|2025-07-05 08:35:19|3b55cb2e-4a2a-449...| Mobile|2025-07-05|
|02576066-f601-4c4...|BB10000434| match_8|        HT/FT|      1376.0| 8.0|2025-07-05 08:35:19|af0ea451-e133-4dd...| Retail|2025-07-05|
|3356dd0a-4b62-421...|BB10001250| match_8|   Over/Under|      1072.0| 5.0|2025-07-05 08:35:19|55d7d635-6798-4c9...| Mobile|2025-07-05|
|8272ad5b-9324-4a3...|BB10003958|match_92|   Over/Under

#### **Step 10: Data Quality Checks**
The script performs data quality checks on the df_cleaned_betting_events DataFrame to validate critical fields such as event_id, match_id, user_id, stake_amount, odds, bet_time, channel, and bet_type. It checks for nulls, invalid categories, future timestamps, and numeric anomalies. Results are aggregated to identify violations before promoting data to the Silver layer.defined thresholds to flag issues.
| Check                | Description                                                          |
| -------------------- | -------------------------------------------------------------------- |
| `null_event_id`      | `event_id` must not be null                                          |
| `null_match_id`      | `match_id` must not be null                                          |
| `null_user_id`       | `user_id` must not be null                                           |
| `null_device_id`     | `device_id` must not be null                                         |
| `stake_non_positive` | `stake_amount` should be a positive value                            |
| `odds_not_gt_one`    | `odds` must be greater than 1 (realistic betting scenario)           |
| `invalid_channel`    | `channel` must be one of: `web`, `mobile`, `pos`                     |
| `invalid_bet_type`   | `bet_type` must be one of: `Full-time Win`, `Draw`, `Over/Under`     |
| `future_bet_time`    | `bet_time` must not be in the future relative to current system time |
| `duplicate_event_id` | Duplicate `event_id` values should not exist in the data             |





In [24]:
from pyspark.sql.functions import col, lower, trim, abs, round, current_timestamp, when, count
from pyspark.sql.types import FloatType, TimestampType

# 1. Reference raw DataFrame
df = df_cleaned_betting_events  # rename for brevity

# 2. Canonical lookups
VALID_BET_TYPES = ["Full‑time Win", "Draw", "Over/Under"]
VALID_CHANNELS  = ["web", "mobile", "pos"]

# 3. Column-level cleansing
df_clean = (
    df
    .withColumn("channel", lower(trim(col("channel"))))
    .withColumn("stake_amount", abs(col("stake_amount").cast(FloatType())))
    .withColumn("stake_amount", round(col("stake_amount"), 0))
    .withColumn("odds", abs(col("odds").cast(FloatType())))
    .withColumn("bet_time", col("bet_time").cast(TimestampType()))
)

# 4. Filter invalid records
df_clean = (
    df_clean
    .filter(
        col("event_id").isNotNull() &
        col("user_id").isNotNull() &
        col("match_id").isNotNull() &
        col("device_id").isNotNull()
    )
    .filter(
        lower(trim(col("bet_type"))).isin([bt.lower() for bt in VALID_BET_TYPES]) &
        col("channel").isin(VALID_CHANNELS)
    )
    .filter(col("odds") > 1)
    .filter(col("bet_time") <= current_timestamp())
)

# 5. De-duplicate event_id
df_clean = df_clean.dropDuplicates(["event_id"])

# 6. Data Quality Report Summary
dq_checks = {
    "stake_non_positive": col("stake_amount") <= 0,
    "odds_not_gt_one": col("odds") <= 1,
    "future_bet_time": col("bet_time") > current_timestamp(),
}

# 7. Compute and Print Results
print("\nBET_RESULTS DATA QUALITY REPORT")
print("--------------------------------------------------")

for check_name, condition in dq_checks.items():
    count_violations = df_cleaned_betting_events.filter(condition).count()
    status = "✅ OK" if count_violations == 0 else "❌ FAIL"
    print(f"{check_name.ljust(28)}: {str(count_violations).rjust(6)}  ->  {status}")

print("--------------------------------------------------")


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 26, Finished, Available, Finished)


BET_RESULTS DATA QUALITY REPORT
--------------------------------------------------
stake_non_positive          :      0  ->  ✅ OK
odds_not_gt_one             :      0  ->  ✅ OK
future_bet_time             :      0  ->  ✅ OK
--------------------------------------------------


#### **Step 11: Data Load to Silver Lakehouse Layer**
This step writes the cleaned users data to the Silver layer of the Lakehouse using Delta format. It uses a MERGE strategy to enable incremental upserts—updating existing user preferences and inserting new ones—while avoiding duplicates.

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/kafka_eventstream/cleaned/cleaned_betting_events.

Prepare Columns for Consistency:
- event_date: Adds the current date for use as a partition column to optimize query performance and simplify incremental loads.
- load_timestamp: Adds a consistent timestamp string (formatted as YYYYMMDDHHMMSS) that represents when the data was loaded.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key user_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by ingestion_date for efficient querying and file organization.


In [31]:
# Define path
silver_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_betting_events"

# Add partitioning and timestamp columns
df_cleaned_betting_events = df_cleaned_betting_events.withColumn("event_date", to_date("event_date"))
df_cleaned_betting_events = df_cleaned_betting_events.withColumn("load_timestamp", lit(datetime.now().strftime("%Y%m%d%H%M%S")))

# Get list of columns (excluding partition and metadata)
columns = [col for col in df_cleaned_betting_events.columns if col != "event_date"]

# Check if Delta table exists
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    # Build insert clause for all columns
    insert_clause = ", ".join([f"source.{c}" for c in df_cleaned_betting_events.columns])

    delta_table.alias("target").merge(
        df_cleaned_betting_events.alias("source"),
        "target.event_id = source.event_id"
    ).whenNotMatchedInsert(
        values={c: f"source.{c}" for c in df_cleaned_betting_events.columns}
    ).execute()
else:
    # First write
    df_cleaned_betting_events.write.format("delta") \
        .mode("append") \
        .partitionBy("event_date") \
        .save(silver_path)


StatementMeta(, dd250417-c814-473e-81c8-d294508e2b58, 33, Finished, Available, Finished)